# Verificaciones del plan de proyecto
**SCY1102 – Gestión de Proyectos de Datos · Evaluación Parcial N°2**
Integrantes: Moira Bittner · Nicolás Carrasco · Martín Corvalán · Martín Naranjo

Este notebook respalda las cifras del informe *Plan de proyecto: modelo predictivo de costos médicos*. No es parte obligatoria del encargo: sirve para responder en la defensa de dónde sale cada número.

| Sección | Qué verifica | Dónde se usa en el informe |
|---|---|---|
| 1 | Calidad de los datos | RD1, H2, supuesto S1, riesgo R1 |
| 2 | Concentración del gasto | Sección 1.1 |
| 3 | Modelos y criterio de RF1 | RF1, entregable 2, H4 |
| 4 | Auditoría de equidad y criterio de RF2 | RF2, H5, riesgo R2 |
| 5 | Exclusión del sexo | RF1, RR1 |
| 6 | Presupuesto | RR2, Tabla 5 |
| 7 | Cronograma e hitos | Tabla 4, Figura 2 |
| 8 | Matriz de riesgos | Tabla 6 |

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df = pd.read_csv('../data/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,"16,884.92"
1,18,male,33.77,1,no,southeast,"1,725.55"
2,28,male,33.00,3,no,southeast,"4,449.46"
3,33,male,22.70,0,no,northwest,"21,984.47"
4,32,male,28.88,0,no,northwest,"3,866.86"


## 1. Calidad de los datos (RD1 · H2)
El informe dice: 1.338 registros, 7 variables, sin nulos, 1 duplicado, edades de 18 a 64 años y regiones de EE. UU.

In [2]:
print('Registros y variables:', df.shape)
print('Valores nulos:', df.isna().sum().sum())
print('Registros duplicados:', df.duplicated().sum())
print('Rango de edad:', df['age'].min(), 'a', df['age'].max(), 'años')
print('Regiones:', sorted(df['region'].unique()))
print('Mayores de 65 años:', (df['age'] >= 65).sum())

Registros y variables: (1338, 7)
Valores nulos: 0
Registros duplicados: 1
Rango de edad: 18 a 64 años
Regiones: ['northeast', 'northwest', 'southeast', 'southwest']
Mayores de 65 años: 0


**Conclusión.** Los datos no tienen mayores de 65 años ni afiliados chilenos. Esto respalda el supuesto S1, el riesgo R1 (nivel crítico) y la condición de salida de construir un conjunto de datos chileno antes de producción.

## 2. Concentración del gasto (sección 1.1)
El informe dice: gasto medio USD 13.270, mediana USD 9.382, el 10 % más costoso reúne el 32,0 % del gasto y un fumador gasta 3,8 veces más que un no fumador.

In [3]:
media, mediana = df['charges'].mean(), df['charges'].median()
print(f'Gasto medio: USD {media:,.0f} | Mediana: USD {mediana:,.0f}')

n_top = int(round(len(df) * 0.10))
orden = df['charges'].sort_values(ascending=False)
print(f'El 10 % más costoso ({n_top} afiliados) reúne el {orden.head(n_top).sum() / orden.sum():.1%} del gasto total')

fum = df.loc[df['smoker'] == 'yes', 'charges'].mean()
no_fum = df.loc[df['smoker'] == 'no', 'charges'].mean()
print(f'Fumador: USD {fum:,.0f} | No fumador: USD {no_fum:,.0f} | Razón: {fum / no_fum:.1f} veces')
print(f'Proporción de fumadores: {(df["smoker"] == "yes").mean():.1%}')

fum_obeso = df.loc[(df['smoker'] == 'yes') & (df['bmi'] >= 30), 'charges'].mean()
print(f'Fumador con IMC ≥ 30: USD {fum_obeso:,.0f}')

decil = df[df['charges'] >= df['charges'].quantile(0.9)]
print(f'Fumadores dentro del decil de mayor costo: {(decil["smoker"] == "yes").mean():.1%}')

Gasto medio: USD 13,270 | Mediana: USD 9,382
El 10 % más costoso (134 afiliados) reúne el 32.0% del gasto total
Fumador: USD 32,050 | No fumador: USD 8,434 | Razón: 3.8 veces
Proporción de fumadores: 20.5%
Fumador con IMC ≥ 30: USD 41,558
Fumadores dentro del decil de mayor costo: 97.8%


## 3. Modelos y criterio de aceptación de RF1 (H4)
Misma configuración del informe de factibilidad: partición 75/25 con `random_state=42`. La línea base es predecir el promedio de entrenamiento para todos.

**Criterio de RF1:** R² ≥ 0,80 y MAE al menos 60 % menor que la línea base.

In [4]:
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True)
y = df['charges']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

lr = LinearRegression().fit(X_tr, y_tr)
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
pred_lr, pred_rf = lr.predict(X_te), rf.predict(X_te)

mae_base = mean_absolute_error(y_te, np.full(len(y_te), y_tr.mean()))
resultados = pd.DataFrame({
    'R²': [np.nan, r2_score(y_te, pred_lr), r2_score(y_te, pred_rf)],
    'MAE (USD)': [mae_base, mean_absolute_error(y_te, pred_lr), mean_absolute_error(y_te, pred_rf)],
}, index=['Línea base (promedio)', 'Regresión lineal', 'Random Forest'])
resultados['Reducción vs. línea base'] = 1 - resultados['MAE (USD)'] / mae_base
resultados

,R²,MAE (USD),Reducción vs. línea base
Línea base (promedio),NaN,"9,253.31",0.00
Regresión lineal,0.77,"4,243.65",0.54
Random Forest,0.85,"2,607.84",0.72


In [5]:
r2_rf = r2_score(y_te, pred_rf)
reduccion = 1 - mean_absolute_error(y_te, pred_rf) / mae_base
print(f'R² del Random Forest: {r2_rf:.3f} → cumple R² ≥ 0,80: {r2_rf >= 0.80}')
print(f'Reducción del MAE: {reduccion:.1%} → cumple reducción ≥ 60 %: {reduccion >= 0.60}')

R² del Random Forest: 0.850 → cumple R² ≥ 0,80: True
Reducción del MAE: 71.8% → cumple reducción ≥ 60 %: True


**Conclusión.** El Random Forest cumple el criterio de RF1 con margen. Los umbrales (0,80 y 60 %) quedaron por debajo del resultado obtenido (0,850 y 71,8 %) para que fueran exigentes pero alcanzables con datos nuevos.

*Nota:* la regresión lineal es determinista y reproduce exactamente las cifras del informe de factibilidad. El Random Forest tiene un componente aleatorio, por lo que sus valores pueden variar unos pocos dólares respecto del informe (por ejemplo, MAE de USD 2.608 frente a USD 2.610); las conclusiones no cambian.

## 4. Auditoría de equidad y criterio de RF2 (H5)
El informe dice: con la regresión lineal, el error en fumadores es 2,6 veces el de no fumadores.

**Criterio de RF2:** la diferencia de MAE entre los subgrupos de cada variable debe ser inferior al 25 %. Se mide como (MAE mayor − MAE menor) / MAE menor.

In [6]:
prueba = X_te.copy()
prueba['real'] = y_te
prueba['pred_lr'], prueba['pred_rf'] = pred_lr, pred_rf
prueba = prueba.join(df[['sex', 'smoker', 'region']])
prueba['tramo_edad'] = pd.cut(prueba['age'], [17, 34, 49, 64], labels=['18–34', '35–49', '50–64'])
prueba['tramo_imc'] = np.where(prueba['bmi'] >= 30, 'IMC ≥ 30', 'IMC < 30')

def mae_por_grupo(col, pred):
    return prueba.groupby(col, observed=True).apply(lambda g: mean_absolute_error(g['real'], g[pred]))

lr_fum = mae_por_grupo('smoker', 'pred_lr')
print(f'Regresión lineal → MAE no fumadores: USD {lr_fum["no"]:,.0f} | fumadores: USD {lr_fum["yes"]:,.0f}')
print(f'El error en fumadores es {lr_fum["yes"] / lr_fum["no"]:.1f} veces el de no fumadores')

Regresión lineal → MAE no fumadores: USD 3,223 | fumadores: USD 8,250
El error en fumadores es 2.6 veces el de no fumadores


In [7]:
filas = []
for var, nombre in [('sex', 'Sexo'), ('tramo_edad', 'Tramo de edad'), ('region', 'Región'),
                    ('smoker', 'Condición de fumador'), ('tramo_imc', 'Tramo de IMC')]:
    for modelo, col in [('Regresión lineal', 'pred_lr'), ('Random Forest', 'pred_rf')]:
        m = mae_por_grupo(var, col)
        brecha = (m.max() - m.min()) / m.min()
        filas.append([nombre, modelo, m.min(), m.max(), brecha, 'Sí' if brecha < 0.25 else 'No'])
auditoria = pd.DataFrame(filas, columns=['Variable', 'Modelo', 'MAE menor', 'MAE mayor', 'Diferencia', 'Cumple < 25 %'])
auditoria.style.format({'MAE menor': 'USD {:,.0f}', 'MAE mayor': 'USD {:,.0f}', 'Diferencia': '{:.0%}'})

,Variable,Modelo,MAE menor,MAE mayor,Diferencia,Cumple < 25 %
0,Sexo,Regresión lineal,"USD 3,755","USD 4,718",26%,No
1,Sexo,Random Forest,"USD 2,482","USD 2,730",10%,Sí
2,Tramo de edad,Regresión lineal,"USD 4,047","USD 4,444",10%,Sí
3,Tramo de edad,Random Forest,"USD 2,127","USD 2,885",36%,No
4,Región,Regresión lineal,"USD 3,499","USD 5,168",48%,No
5,Región,Random Forest,"USD 2,531","USD 2,823",12%,Sí
6,Condición de fumador,Regresión lineal,"USD 3,223","USD 8,250",156%,No
7,Condición de fumador,Random Forest,"USD 2,586","USD 2,694",4%,Sí
8,Tramo de IMC,Regresión lineal,"USD 3,626","USD 4,721",30%,No
9,Tramo de IMC,Random Forest,"USD 2,432","USD 2,744",13%,Sí


In [8]:
sesgo = prueba['pred_rf'] - prueba['real']
print(f'Sesgo medio del Random Forest en fumadores: USD {sesgo[prueba["smoker"] == "yes"].mean():+,.0f}')
print(f'Sesgo medio del Random Forest en southeast: USD {sesgo[prueba["region"] == "southeast"].mean():+,.0f}')

Sesgo medio del Random Forest en fumadores: USD +964
Sesgo medio del Random Forest en southeast: USD +723


**Conclusión.** La regresión lineal no aprobaría H5: su error en fumadores es 2,6 veces mayor. Por eso RF2 es *Must have* y H5 funciona como puerta de aprobación: un R² global alto no basta si el modelo se equivoca más con un grupo. El Random Forest cumple en sexo, región, condición de fumador y tramo de IMC, pero **no cumple en tramo de edad** (diferencia de 36 %): se equivoca más con los afiliados de 50 a 64 años, que son los de mayor gasto. Con el prototipo actual, H5 no se aprobaría todavía. Esto no contradice el plan, lo confirma: es exactamente el caso del riesgo R2, y la holgura de las semanas 10 a 15 existe para iterar (por ejemplo, con transformación logarítmica del costo) antes de la decisión go/no-go de H7.

## 5. Exclusión del sexo (RF1 · RR1)
El informe dice: excluir el sexo no reduce la precisión.

In [9]:
cols_sin_sexo = [c for c in X.columns if c != 'sex_male']
rf_sin_sexo = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_tr[cols_sin_sexo], y_tr)
r2_sin = r2_score(y_te, rf_sin_sexo.predict(X_te[cols_sin_sexo]))
print(f'R² con sexo: {r2_rf:.4f} | R² sin sexo: {r2_sin:.4f} | Diferencia: {r2_sin - r2_rf:+.4f}')

R² con sexo: 0.8502 | R² sin sexo: 0.8530 | Diferencia: +0.0029


**Conclusión.** Sin la variable sexo el modelo predice igual o marginalmente mejor. Cumplir la norma no cuesta precisión.

## 6. Presupuesto (RR2 · Tabla 5)

In [10]:
dolar = 929.18
rrhh = {
    'Data Scientist (completa meses 1–4)': 3_500_000 * 4,
    'Data Scientist (media jornada meses 5–12)': 1_750_000 * 8,
    'Data Engineer (completa meses 1–4)': 3_400_000 * 4,
    'Data Analyst (media jornada meses 5–12)': 900_000 * 8,
}
for k, v in rrhh.items():
    print(f'{k:45s} CLP {v:>12,.0f}')
total_rrhh = sum(rrhh.values())
print(f'{"Total recursos humanos":45s} CLP {total_rrhh:>12,.0f}')

infra = {'SageMaker': 188.90, 'S3': 4.15, 'Power BI (5 usuarios)': 70.00, 'Bedrock': 30.00}
total_infra = sum(v * 12 for v in infra.values())
print(f'\nInfraestructura, licencias y agentes (12 meses): USD {total_infra:,.2f}')
total_usd = total_rrhh / dolar + total_infra
print(f'Total año 1: USD {total_usd:,.0f}  (CLP {total_rrhh + total_infra * dolar:,.0f})')
print(f'Peso de los costos en USD: {total_infra / total_usd:.1%}  → base del riesgo R9')

Data Scientist (completa meses 1–4)           CLP   14,000,000
Data Scientist (media jornada meses 5–12)     CLP   14,000,000
Data Engineer (completa meses 1–4)            CLP   13,600,000
Data Analyst (media jornada meses 5–12)       CLP    7,200,000
Total recursos humanos                        CLP   48,800,000

Infraestructura, licencias y agentes (12 meses): USD 3,516.60
Total año 1: USD 56,036  (CLP 52,067,554)
Peso de los costos en USD: 6.3%  → base del riesgo R9


## 7. Cronograma e hitos (Tabla 4 · Figura 2)

In [11]:
fases = {'1. Comprensión del negocio': (1, 2), '2. Comprensión de los datos': (3, 3),
         '3. Preparación de los datos': (4, 4), '4. Modelado': (5, 6),
         '5. Evaluación': (7, 7), '6. Despliegue': (8, 9)}
efectivo = sum(fin - ini + 1 for ini, fin in fases.values())
print('Semanas de trabajo efectivo:', efectivo)
print('Holgura (semanas 10 a 15):', 15 - 10 + 1)
print('Desarrollo total:', efectivo + 6 + 1, 'semanas = 4 meses (1 mes = 4 semanas)')
print('Operación del piloto:', 12 - 4, 'meses')

Semanas de trabajo efectivo: 9
Holgura (semanas 10 a 15): 6
Desarrollo total: 16 semanas = 4 meses (1 mes = 4 semanas)
Operación del piloto: 8 meses


## 8. Matriz de riesgos (Tabla 6)
Regla: nivel = P × I, donde 1–2 es bajo, 3–4 medio, 6 alto y 9 crítico.

In [12]:
riesgos = pd.DataFrame([
    ('R1', 'Datos no representativos', 3, 3), ('R2', 'Error desigual entre subgrupos', 3, 2),
    ('R3', 'Uso para tarificar', 2, 3), ('R4', 'Datos de salud sin base legal', 2, 3),
    ('R5', 'Salida de un perfil clave', 2, 3), ('R6', 'Beneficio no se materializa', 2, 3),
    ('R7', 'Retraso en aprobaciones', 2, 2), ('R8', 'Sobreconsumo cloud', 2, 2),
    ('R9', 'Tipo de cambio', 2, 1)], columns=['ID', 'Riesgo', 'P', 'I'])
riesgos['P × I'] = riesgos['P'] * riesgos['I']
riesgos['Nivel'] = pd.cut(riesgos['P × I'], [0, 2, 4, 6, 9], labels=['Bajo', 'Medio', 'Alto', 'Crítico'])
riesgos

,ID,Riesgo,P,I,P × I,Nivel
0,R1,Datos no representativos,3,3,9,Crítico
1,R2,Error desigual entre subgrupos,3,2,6,Alto
2,R3,Uso para tarificar,2,3,6,Alto
3,R4,Datos de salud sin base legal,2,3,6,Alto
4,R5,Salida de un perfil clave,2,3,6,Alto
5,R6,Beneficio no se materializa,2,3,6,Alto
6,R7,Retraso en aprobaciones,2,2,4,Medio
7,R8,Sobreconsumo cloud,2,2,4,Medio
8,R9,Tipo de cambio,2,1,2,Bajo


## Resumen
Las cifras del informe se reproducen con el conjunto de datos entregado. Los cálculos de presupuesto, cronograma y riesgos son consistentes con las tablas del plan.